## Set-up the API Key 

In [ ]:
from together import Together
import os
import requests
import json

# Get a free API key from https://api.together.xyz/settings/api-keys
os.environ["TOGETHER_API_KEY"] = "XXXXXXXX" #

def llama32(messages, model_size=11):
  model = f"meta-llama/Llama-3.2-{model_size}B-Vision-Instruct-Turbo"
  url = "https://api.together.xyz/v1/chat/completions"
  payload = {
    "model": model,
    "max_tokens": 4096,
    "temperature": 0.0,
    "stop": ["<|eot_id|>","<|eom_id|>"],
    "messages": messages
  }

  headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Authorization": "Bearer " + os.environ["TOGETHER_API_KEY"]
  }
  res = json.loads(requests.request("POST", url, headers=headers, data=json.dumps(payload)).content)

  if 'error' in res:
    raise Exception(res['error'])

  return res['choices'][0]['message']['content']

## The Main Code

In [ ]:
import os
import pandas as pd
import csv
import base64
import re
import json
from tqdm import tqdm  # ✅ Progress bar for tracking

# 📂 Folder containing all images
image_folder = "XXXXXX"

# 📄 Output CSV file (keeps adding new data)
csv_filename = "XXXXX"

# ✅ Configure image ID format
keep_extension = True  # Set to False if you want image IDs without .png/.jpg

# 🏷️ Define categories for Q&A
categories = ["Abnormality", "Severity", "Location", "Diagnostic", "Reasoning", "Treatment"]

# ✅ Function to encode image to Base64
def encode_image(image_path):
    """Convert image to Base64 encoding."""
    with open(image_path, "rb") as img:
        return base64.b64encode(img.read()).decode('utf-8')

# ✅ Get list of all images in folder
image_files = [f for f in os.listdir(image_folder) if f.endswith((".png", ".jpg", ".jpeg"))]
total_images = len(image_files)  # Count total images

# ✅ Open CSV file in append mode
with open(csv_filename, mode="a", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["image_id", "category", "question", "answer"])
    
    # Write headers only if the file is empty
    if os.stat(csv_filename).st_size == 0:
        writer.writeheader()

    # 🚀 Process each image with a progress bar
    with tqdm(total=total_images, desc="Processing Images", unit="image", dynamic_ncols=True) as pbar:
        for image_file in image_files:
            image_id = image_file if keep_extension else os.path.splitext(image_file)[0]

            # 🖼️ Convert image to Base64
            image_path = os.path.join(image_folder, image_file)
            base64_image = encode_image(image_path)

            # 📝 Define the AI prompt
            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"""
                            You are a medical AI assistant analyzing an X-ray image. Your task is to examine the image and generate structured question-answer pairs.
                            Analyze the given image ({image_id}), and if there is no annotation, that means there is no abnormality in the image. 
                            If there are abnormalities, generate six explicitly written questions and their respective answers, considering all detected abnormalities together.
                             The categories are:

                            1. Abnormality: Write a question about the abnormalities and provide a detailed response.  
                            2. Severity: Write a question about the severity of the abnormalities and provide an answer.  
                            3. Location: Write a question about the location of the abnormalities and describe their placement in the image.  
                            4. Diagnostic: Write a question asking for the most likely diagnosis and provide a single diagnostic explanation.  
                            5. Reasoning: Write a question asking for the possible causes of the abnormalities and provide a response.  
                            6. Treatment: Write a question about the suggested treatment or management options for the detected abnormalities and provide a relevant answer.  

                            If there is no abnormality detected, generate only **one** question-answer pair stating:  
                            - **Question:** "Is there any abnormality present in this image?"  
                            - **Answer:** "No abnormality detected."

                            **Rules:**  
                            - Each question must be explicitly written, followed by its corresponding answer.  
                            - Do not repeat the question inside the answer.  
                            - Do not include question labels like "Question 2: Severity".  
                            - Keep the responses structured and professional.  
                            """
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]

            # ✅ Send request to AI model
            try:
                raw_result = llama32(messages)  # Get response from AI model
                result = raw_result.strip()
            except Exception as e:
                result = f"Error processing image: {str(e)}"

            # 🚀 Process the AI response into Q&A pairs
            qa_pairs = []

            # ✅ Remove text after any "Question" word in the answer
            def clean_answer(answer):
                """Cleans AI-generated answers to remove unnecessary 'Question XYZ' texts."""
                cleaned = re.sub(r"Question\s*(Abnormality|Severity|Location|Diagnostic|Reasoning|Treatment)\b", "", answer, flags=re.IGNORECASE).strip()
                return cleaned

            # ✅ Ensure abnormalities are detected correctly
            if "Error processing image" in result:
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Error",
                    "question": "Could not process this image.",
                    "answer": result
                })
            elif "No abnormality detected." in result:
                # ✅ Handle case when no abnormality is detected
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Abnormality",
                    "question": "Is there any abnormality present in this image?",
                    "answer": "No abnormality detected."
                })
            else:
                # ✅ Process multiple Q&A pairs for abnormal images
                lines = result.split("\n")
                current_question = None
                current_answer = []
                category_index = 0  # Track question category

                for line in lines:
                    line = line.strip()
                    
                    # ✅ Clean up formatting (remove **, *, _ symbols)
                    line = re.sub(r"[*_]+", "", line).strip()
                    line = re.sub(r"\b\d+:", "", line).strip()  # Remove numbers before questions

                    # ✅ Detect questions (starting with "What", "Where", "How", "Why", "Is", "Are")
                    if line.lower().startswith(("what", "where", "how", "why", "is", "are")):
                        # Save the previous Q&A pair if there's an existing question
                        if current_question and current_answer:
                            final_answer = clean_answer(" ".join(current_answer).strip())
                            qa_pairs.append({
                                "image_id": image_id,
                                "category": categories[category_index] if category_index < len(categories) else "Other",
                                "question": current_question,
                                "answer": final_answer if final_answer else "No answer provided."
                            })
                            category_index += 1  
                        current_question = line  # Set the new question
                        current_answer = []
                    elif current_question:
                        current_answer.append(line)

                # ✅ Save the last Q&A pair
                if current_question and current_answer:
                    final_answer = clean_answer(" ".join(current_answer).strip())
                    qa_pairs.append({
                        "image_id": image_id,
                        "category": categories[category_index] if category_index < len(categories) else "Other",
                        "question": current_question,
                        "answer": final_answer if final_answer else "No answer provided."
                    })

            # ✅ **Save results immediately after processing each image**
            for qa in qa_pairs:
                writer.writerow(qa)
                csv_file.flush()  # 🔹 Immediate write to CSV file

            # ✅ Update progress bar silently (only shows total progress)
            pbar.update(1)

print(f"✅ All Q&A pairs saved incrementally to {csv_filename} successfully!")


## CODE For Generating Train-Set QA pair

In [ ]:
import os
import pandas as pd
import csv
import base64
import re
import json
import time
from tqdm import tqdm

# 📂 Image folder
image_folder = "XXXXXX"

# 📄 Old CSV (already processed)
old_csv_filename = "XXXXXXXX"

# 📄 New CSV (new results)
new_csv_filename = "XXXXXXXXXX"

# Categories
categories = ["Abnormality", "Severity", "Location", "Diagnostic", "Reasoning", "Treatment"]

# ✅ Encode image to base64
def encode_image(image_path):
    with open(image_path, "rb") as img:
        return base64.b64encode(img.read()).decode('utf-8')

# ✅ Retry-safe model call
def call_llama32_with_retry(messages, max_retries=5, backoff=5):
    for attempt in range(max_retries):
        try:
            raw = llama32(messages)
            return raw.strip()
        except Exception as e:
            print(f"⚠️ Retry {attempt+1}/{max_retries} failed: {e}")
            time.sleep(backoff * (attempt + 1))
    return "Error processing image: Max retries exceeded."

# ✅ Load processed image IDs
processed_images = set()
if os.path.exists(old_csv_filename) and os.path.getsize(old_csv_filename) > 0:
    df = pd.read_csv(old_csv_filename)
    processed_images.update(df["image_id"].astype(str).tolist())

# ✅ List images
image_files = [f for f in os.listdir(image_folder) if f.endswith((".png", ".jpg", ".jpeg"))]
unprocessed_images = [img for img in image_files if img not in processed_images]

# ✅ Start writing to new CSV
with open(new_csv_filename, mode="w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["image_id", "category", "question", "answer"])
    writer.writeheader()

    with tqdm(total=len(unprocessed_images), desc="Processing New Images", unit="image", dynamic_ncols=True) as pbar:
        for image_file in unprocessed_images:
            image_id = image_file
            image_path = os.path.join(image_folder, image_file)
            base64_image = encode_image(image_path)

            # 📩 Prompt
            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"""
You are a medical AI assistant analyzing an X-ray image. Your task is to examine the image and generate structured question-answer pairs.
Analyze the given image ({image_id}), and if there is no annotation, that means there is no abnormality in the image.
If there are abnormalities, generate six explicitly written questions and their respective answers, considering all detected abnormalities together. The categories are:

1. Abnormality: Write a question about the abnormalities and provide a detailed response.  
2. Severity: Write a question about the severity of the abnormalities and provide an answer.  
3. Location: Write a question about the location of the abnormalities and describe their placement in the image.  
4. Diagnostic: Write a question asking for the most likely diagnosis and provide a single diagnostic explanation.  
5. Reasoning: Write a question asking for the possible causes of the abnormalities and provide a response.  
6. Treatment: Write a question about the suggested treatment or management options for the detected abnormalities and provide a relevant answer.  

If there is no abnormality detected, generate only one question-answer pair:  
- Question: "Is there any abnormality present in this image?"  
- Answer: "No abnormality detected."

Rules:
- Do not repeat the question inside the answer.
- Do not include labels like "Question 2: Severity".
- Keep responses structured and professional.
"""
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]

            # 🔁 Call model with retry logic
            result = call_llama32_with_retry(messages)

            qa_pairs = []

            def clean_answer(answer):
                return re.sub(r"Question\s*(Abnormality|Severity|Location|Diagnostic|Reasoning|Treatment)?\b", "", answer, flags=re.IGNORECASE).strip()

            # ✅ No abnormality case
            if "No abnormality detected." in result:
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Abnormality",
                    "question": "Is there any abnormality present in this image?",
                    "answer": "No abnormality detected."
                })
            elif "Error processing image" in result or not result:
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Error",
                    "question": "Could not process this image.",
                    "answer": result
                })
            else:
                lines = result.split("\n")
                current_question = None
                current_answer = []
                category_index = 0

                for line in lines:
                    line = re.sub(r"[*_]+", "", line).strip()
                    line = re.sub(r"\b\d+:", "", line)

                    if line.lower().startswith(("what", "where", "how", "why", "is", "are")):
                        if current_question and current_answer:
                            final_answer = clean_answer(" ".join(current_answer).strip())
                            qa_pairs.append({
                                "image_id": image_id,
                                "category": categories[category_index] if category_index < len(categories) else "Other",
                                "question": current_question,
                                "answer": final_answer
                            })
                            category_index += 1
                        current_question = line
                        current_answer = []
                    elif current_question:
                        current_answer.append(line)

                if current_question and current_answer:
                    final_answer = clean_answer(" ".join(current_answer).strip())
                    qa_pairs.append({
                        "image_id": image_id,
                        "category": categories[category_index] if category_index < len(categories) else "Other",
                        "question": current_question,
                        "answer": final_answer
                    })

            # ✅ Save to CSV
            if qa_pairs:
                for qa in qa_pairs:
                    writer.writerow(qa)
                csv_file.flush()
            else:
                print(f"⚠️ Skipped empty output: {image_id}")

            pbar.update(1)

print(f"✅ Completed. All unprocessed images saved to: {new_csv_filename}")


In [ ]:
import os
import pandas as pd
import csv
import base64
import re
import json
from tqdm import tqdm  # ✅ Progress bar for tracking

# 📂 Folder containing all images
image_folder = "XXXXXX"

# 📄 Output CSV file (keeps adding new data)
csv_filename = "XXXXX"

# ✅ Configure image ID format
keep_extension = True  # Set to False if you want image IDs without .png/.jpg

# 🏷️ Define categories for Q&A
categories = ["Abnormality", "Severity", "Location", "Diagnostic", "Reasoning", "Treatment"]

# ✅ Function to encode image to Base64
def encode_image(image_path):
    """Convert image to Base64 encoding."""
    with open(image_path, "rb") as img:
        return base64.b64encode(img.read()).decode('utf-8')

# ✅ Get list of all images in folder
image_files = [f for f in os.listdir(image_folder) if f.endswith((".png", ".jpg", ".jpeg"))]
total_images = len(image_files)  # Count total images

# ✅ Open CSV file in append mode
with open(csv_filename, mode="a", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["image_id", "category", "question", "answer"])
    
    # Write headers only if the file is empty
    if os.stat(csv_filename).st_size == 0:
        writer.writeheader()

    # 🚀 Process each image with a progress bar
    with tqdm(total=total_images, desc="Processing Images", unit="image", dynamic_ncols=True) as pbar:
        for image_file in image_files:
            image_id = image_file if keep_extension else os.path.splitext(image_file)[0]

            # 🖼️ Convert image to Base64
            image_path = os.path.join(image_folder, image_file)
            base64_image = encode_image(image_path)

            # 📝 Define the AI prompt
            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"""
                            You are a medical AI assistant analyzing an X-ray image. Your task is to examine the image and generate structured question-answer pairs.
                            Analyze the given image ({image_id}), and if there is no annotation, that means there is no abnormality in the image.
                            If there are abnormalities, generate six explicitly written questions and their respective answers, considering all detected abnormalities together. The categories are:

                            1. Abnormality: Write a question about the abnormalities and provide a detailed response.  
                            2. Severity: Write a question about the severity of the abnormalities and provide an answer.  
                            3. Location: Write a question about the location of the abnormalities and describe their placement in the image.  
                            4. Diagnostic: Write a question asking for the most likely diagnosis and provide a single diagnostic explanation.  
                            5. Reasoning: Write a question asking for the possible causes of the abnormalities and provide a response.  
                            6. Treatment: Write a question about the suggested treatment or management options for the detected abnormalities and provide a relevant answer.  

                            If there is no abnormality detected, generate only **one** question-answer pair stating:  
                            - **Question:** "Is there any abnormality present in this image?"  
                            - **Answer:** "No abnormality detected."

                            **Rules:**  
                            - Each question must be explicitly written, followed by its corresponding answer.  
                            - Do not repeat the question inside the answer.  
                            - Do not include question labels like "Question 2: Severity".  
                            - Keep the responses structured and professional.  
                            """
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]

            # ✅ Send request to AI model
            try:
                raw_result = llama32(messages)  # Get response from AI model
                result = raw_result.strip()
            except Exception as e:
                result = f"Error processing image: {str(e)}"

            # 🚀 Process the AI response into Q&A pairs
            qa_pairs = []

            # ✅ Remove text after any "Question" word in the answer
            def clean_answer(answer):
                """Cleans AI-generated answers to remove unnecessary 'Question XYZ' texts."""
                cleaned = re.sub(r"Question\s*(Abnormality|Severity|Location|Diagnostic|Reasoning|Treatment)\b", "", answer, flags=re.IGNORECASE).strip()
                return cleaned

            # ✅ Ensure abnormalities are detected correctly
            if "Error processing image" in result:
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Error",
                    "question": "Could not process this image.",
                    "answer": result
                })
            elif "No abnormality detected." in result:
                # ✅ Handle case when no abnormality is detected
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Abnormality",
                    "question": "Is there any abnormality present in this image?",
                    "answer": "No abnormality detected."
                })
            else:
                # ✅ Process multiple Q&A pairs for abnormal images
                lines = result.split("\n")
                current_question = None
                current_answer = []
                category_index = 0  # Track question category

                for line in lines:
                    line = line.strip()
                    
                    # ✅ Clean up formatting (remove **, *, _ symbols)
                    line = re.sub(r"[*_]+", "", line).strip()
                    line = re.sub(r"\b\d+:", "", line).strip()  # Remove numbers before questions

                    # ✅ Detect questions (starting with "What", "Where", "How", "Why", "Is", "Are")
                    if line.lower().startswith(("what", "where", "how", "why", "is", "are")):
                        # Save the previous Q&A pair if there's an existing question
                        if current_question and current_answer:
                            final_answer = clean_answer(" ".join(current_answer).strip())
                            qa_pairs.append({
                                "image_id": image_id,
                                "category": categories[category_index] if category_index < len(categories) else "Other",
                                "question": current_question,
                                "answer": final_answer if final_answer else "No answer provided."
                            })
                            category_index += 1  
                        current_question = line  # Set the new question
                        current_answer = []
                    elif current_question:
                        current_answer.append(line)

                # ✅ Save the last Q&A pair
                if current_question and current_answer:
                    final_answer = clean_answer(" ".join(current_answer).strip())
                    qa_pairs.append({
                        "image_id": image_id,
                        "category": categories[category_index] if category_index < len(categories) else "Other",
                        "question": current_question,
                        "answer": final_answer if final_answer else "No answer provided."
                    })

            # ✅ **Save results immediately after processing each image**
            for qa in qa_pairs:
                writer.writerow(qa)
                csv_file.flush()  # 🔹 Immediate write to CSV file

            # ✅ Update progress bar silently (only shows total progress)
            pbar.update(1)

print(f"✅ All Q&A pairs saved incrementally to {csv_filename} successfully!")

## Code for generating Q_A pair for test set 


In [ ]:
import os
import pandas as pd
import csv
import base64
import re
import json
from tqdm import tqdm  # ✅ Progress bar for tracking

# 📂 Folder containing all images
image_folder = "XXXXXX"

# 📄 Output CSV file (keeps adding new data)
csv_filename = "XXXXX"

# ✅ Configure image ID format
keep_extension = True  # Set to False if you want image IDs without .png/.jpg

# 🏷️ Define categories for Q&A
categories = ["Abnormality", "Severity", "Location", "Diagnostic", "Reasoning", "Treatment"]

# ✅ Function to encode image to Base64
def encode_image(image_path):
    """Convert image to Base64 encoding."""
    with open(image_path, "rb") as img:
        return base64.b64encode(img.read()).decode('utf-8')

# ✅ Get list of all images in folder
image_files = [f for f in os.listdir(image_folder) if f.endswith((".png", ".jpg", ".jpeg"))]
total_images = len(image_files)  # Count total images

# ✅ Open CSV file in append mode
with open(csv_filename, mode="a", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=["image_id", "category", "question", "answer"])
    
    # Write headers only if the file is empty
    if os.stat(csv_filename).st_size == 0:
        writer.writeheader()

    # 🚀 Process each image with a progress bar
    with tqdm(total=total_images, desc="Processing Images", unit="image", dynamic_ncols=True) as pbar:
        for image_file in image_files:
            image_id = image_file if keep_extension else os.path.splitext(image_file)[0]

            # 🖼️ Convert image to Base64
            image_path = os.path.join(image_folder, image_file)
            base64_image = encode_image(image_path)

            # 📝 Define the AI prompt
            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"""
                            You are a medical AI assistant analyzing an X-ray image. Your task is to examine the image and generate structured question-answer pairs.
                            Analyze the given image ({image_id}), and if there is no annotation, that means there is no abnormality in the image.
                            If there are abnormalities, generate six explicitly written questions and their respective answers, considering all detected abnormalities together. The categories are:

                            1. Abnormality: Write a question about the abnormalities and provide a detailed response.  
                            2. Severity: Write a question about the severity of the abnormalities and provide an answer.  
                            3. Location: Write a question about the location of the abnormalities and describe their placement in the image.  
                            4. Diagnostic: Write a question asking for the most likely diagnosis and provide a single diagnostic explanation.  
                            5. Reasoning: Write a question asking for the possible causes of the abnormalities and provide a response.  
                            6. Treatment: Write a question about the suggested treatment or management options for the detected abnormalities and provide a relevant answer.  

                            If there is no abnormality detected, generate only **one** question-answer pair stating:  
                            - **Question:** "Is there any abnormality present in this image?"  
                            - **Answer:** "No abnormality detected."

                            Rules:

                            Avoid starting any question with the following words: what, where, which, is, or are.

                            Use diverse, professional clinical phrasing that maintains clarity and avoids redundancy.

                            Ensure each question is explicitly written and followed by its corresponding answer.

                            Do not repeat the question's wording in the answer.

                            Avoid including question labels like "Question 1: Abnormality."

                            Keep the language professional, concise, and structured. 
                            """
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]

            # ✅ Send request to AI model
            try:
                raw_result = llama32(messages)  # Get response from AI model
                result = raw_result.strip()
            except Exception as e:
                result = f"Error processing image: {str(e)}"

            # 🚀 Process the AI response into Q&A pairs
            qa_pairs = []

            # ✅ Remove text after any "Question" word in the answer
            def clean_answer(answer):
                """Cleans AI-generated answers to remove unnecessary 'Question XYZ' texts."""
                cleaned = re.sub(r"Question\s*(Abnormality|Severity|Location|Diagnostic|Reasoning|Treatment)\b", "", answer, flags=re.IGNORECASE).strip()
                return cleaned

            # ✅ Ensure abnormalities are detected correctly
            if "Error processing image" in result:
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Error",
                    "question": "Could not process this image.",
                    "answer": result
                })
            elif "No abnormality detected." in result:
                # ✅ Handle case when no abnormality is detected
                qa_pairs.append({
                    "image_id": image_id,
                    "category": "Abnormality",
                    "question": "Is there any abnormality present in this image?",
                    "answer": "No abnormality detected."
                })
            else:
                # ✅ Process multiple Q&A pairs for abnormal images
                lines = result.split("\n")
                current_question = None
                current_answer = []
                category_index = 0  # Track question category

                for line in lines:
                    line = line.strip()
                    
                    # ✅ Clean up formatting (remove **, *, _ symbols)
                    line = re.sub(r"[*_]+", "", line).strip()
                    line = re.sub(r"\b\d+:", "", line).strip()  # Remove numbers before questions

                    # ✅ Detect questions (starting with "What", "Where", "How", "Why", "Is", "Are")
                    if line.lower().startswith(("what", "where", "how", "why", "is", "are")):
                        # Save the previous Q&A pair if there's an existing question
                        if current_question and current_answer:
                            final_answer = clean_answer(" ".join(current_answer).strip())
                            qa_pairs.append({
                                "image_id": image_id,
                                "category": categories[category_index] if category_index < len(categories) else "Other",
                                "question": current_question,
                                "answer": final_answer if final_answer else "No answer provided."
                            })
                            category_index += 1  
                        current_question = line  # Set the new question
                        current_answer = []
                    elif current_question:
                        current_answer.append(line)

                # ✅ Save the last Q&A pair
                if current_question and current_answer:
                    final_answer = clean_answer(" ".join(current_answer).strip())
                    qa_pairs.append({
                        "image_id": image_id,
                        "category": categories[category_index] if category_index < len(categories) else "Other",
                        "question": current_question,
                        "answer": final_answer if final_answer else "No answer provided."
                    })

            # ✅ **Save results immediately after processing each image**
            for qa in qa_pairs:
                writer.writerow(qa)
                csv_file.flush()  # 🔹 Immediate write to CSV file

            # ✅ Update progress bar silently (only shows total progress)
            pbar.update(1)

print(f"✅ All Q&A pairs saved incrementally to {csv_filename} successfully!")
